# Figure 4 — LLM baseline (OpenAI / Gemini)

## Standalone reviewer notebook

This notebook exposes the live proprietary-LLM inference path used for the Figure 4 baseline without publishing credentials or stored private outputs. It is independent of the TGCM model, checkpoints, and the other reproduction notebooks.

### Environment

- Python 3.12
- `openai`
- `google-genai`
- CPU only; CUDA is not required.

Install the provider SDKs with `pip install -U openai google-genai`.

### Credentials

- OpenAI: set `OPENAI_API_KEY` in the shell/environment.
- Gemini API: set `GEMINI_API_KEY`; alternatively Vertex AI can use `GOOGLE_CLOUD_PROJECT` plus Application Default Credentials.
- **Do not paste API keys into this notebook.**

### Models

- The paper reports the OpenAI baseline as **ChatGPT-5.5** and the Gemini baseline as **Gemini-3**.
- `OPENAI_MODEL` and `GEMINI_MODEL` are environment-overridable so a reviewer can select the exact provider model/snapshot available to their account.
- The defaults below follow the released artifact configuration where possible.

### Data

No training data or checkpoint is required for a live call. The exact Figure 4 prompt is downloaded automatically from the public TGCM artifact. Replace `sequence` with any mixed ATT&CK-technique sequence, or iterate over your own evaluation records.


In [ ]:
from pathlib import Path
import json
import os
import urllib.request

PROMPT_URL = "https://raw.githubusercontent.com/Irish-kw/TGCM_Website/main/reproduction/paper_metadata/figure04_llm_prompt.txt"
PROMPT_FILE = Path.cwd() / "figure04_llm_prompt.txt"
if not PROMPT_FILE.is_file():
    urllib.request.urlretrieve(PROMPT_URL, PROMPT_FILE)
PROMPT_TEMPLATE = PROMPT_FILE.read_text(encoding="utf-8")

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3-flash-preview")

print("prompt:", PROMPT_FILE)
print("OpenAI model:", OPENAI_MODEL)
print("Gemini model:", GEMINI_MODEL)


In [ ]:
def build_prompt(sequence):
    return PROMPT_TEMPLATE.format(question=json.dumps(list(sequence), ensure_ascii=False))

def parse_json_response(text):
    text = text.strip()
    if text.startswith("```"):
        lines = text.splitlines()
        text = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])
    payload = json.loads(text)
    if "locations" not in payload:
        raise ValueError("LLM response does not contain 'locations'.")
    locations = [int(x) for x in payload["locations"]]
    if any(x <= 0 for x in locations):
        raise ValueError("locations must contain positive campaign IDs only.")
    return payload

def validate_response(sequence, payload):
    if len(payload["locations"]) != len(sequence):
        raise ValueError("locations length does not match the input sequence.")
    return payload


In [ ]:
def call_openai(sequence, model=OPENAI_MODEL):
    from openai import OpenAI
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("Set OPENAI_API_KEY in the environment before making a live request.")
    client = OpenAI()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": build_prompt(sequence)}],
    )
    payload = parse_json_response(response.choices[0].message.content)
    return validate_response(sequence, payload)


In [ ]:
def call_gemini(sequence, model=GEMINI_MODEL):
    from google import genai
    from google.genai import types

    api_key = os.getenv("GEMINI_API_KEY")
    project = os.getenv("GOOGLE_CLOUD_PROJECT")
    location = os.getenv("GOOGLE_CLOUD_LOCATION", "global")
    if api_key:
        client = genai.Client(api_key=api_key)
    elif project:
        client = genai.Client(vertexai=True, project=project, location=location)
    else:
        raise RuntimeError("Set GEMINI_API_KEY, or configure GOOGLE_CLOUD_PROJECT + ADC for Vertex AI.")

    response = client.models.generate_content(
        model=model,
        contents=build_prompt(sequence),
        config=types.GenerateContentConfig(response_mime_type="application/json"),
    )
    payload = parse_json_response(response.text)
    return validate_response(sequence, payload)


## Run one example

Choose exactly one provider call below. Live calls incur the normal provider API cost. The notebook intentionally does not execute automatically and stores no API response in the `.ipynb` file.


In [ ]:
sequence = ["T1566", "T1566", "T1486", "T1059", "T1057", "T1003", "T1041"]

# OpenAI:
# result = call_openai(sequence)

# Gemini:
# result = call_gemini(sequence)

# result


## Optional batch loop

For a complete evaluation, replace `sequences` with the fixed Figure 4 evaluation records and collect each returned `locations` vector. Ground-truth labels are used only for downstream metric computation; they are never included in the LLM prompt.


In [ ]:
# Example only — uncomment after selecting a provider.
# sequences = [sequence]
# records = []
# for seq in sequences:
#     response = call_openai(seq)  # or call_gemini(seq)
#     records.append({"sequence": seq, "locations": response["locations"], "response": response})
# Path("llm_live_responses.json").write_text(json.dumps(records, indent=2), encoding="utf-8")
